
# 08. Benchmarking, Keyboard Engine, Advanced Decoding, and Deployment

## Objectives
- Build production-style inference API
- Compare decoding strategies (beam, temperature, top-k, top-p)
- Simulate smartphone keyboard suggestion bar
- Prepare Streamlit deployment workflow


In [ ]:

from pathlib import Path
import json
import pandas as pd
import torch

from utils.keyboard_engine import PredictiveKeyboardEngine
from utils.models import LSTM_LM, StackedLSTM_LM, BiLSTM_LM, GRU_LM, CNN_LSTM_LM, TransformerLM
from utils.vocabulary import Vocabulary

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RESULTS = PROJECT_ROOT / "outputs" / "results"

leaderboard_path = sorted(RESULTS.glob("leaderboard_*.csv"))[-1]
registry_path = sorted(RESULTS.glob("model_registry_*.json"))[-1]

leaderboard = pd.read_csv(leaderboard_path)
registry = json.loads(registry_path.read_text(encoding="utf-8"))

leaderboard.head(10)


In [ ]:

MODEL_BUILDERS = {
    "LSTM": lambda c: LSTM_LM(vocab_size=int(c["vocab_size"]), embedding_dim=int(c["embedding_dim"]), hidden_dim=int(c["hidden_dim"]), num_layers=1),
    "StackedLSTM": lambda c: StackedLSTM_LM(vocab_size=int(c["vocab_size"]), embedding_dim=int(c["embedding_dim"]), hidden_dim=int(c["hidden_dim"]), num_layers=2),
    "BiLSTM": lambda c: BiLSTM_LM(vocab_size=int(c["vocab_size"]), embedding_dim=int(c["embedding_dim"]), hidden_dim=max(int(c["hidden_dim"])//2, 64), num_layers=2),
    "GRU": lambda c: GRU_LM(vocab_size=int(c["vocab_size"]), embedding_dim=int(c["embedding_dim"]), hidden_dim=int(c["hidden_dim"]), num_layers=2),
    "CNN_LSTM": lambda c: CNN_LSTM_LM(vocab_size=int(c["vocab_size"]), embedding_dim=int(c["embedding_dim"]), hidden_dim=int(c["hidden_dim"]), num_filters=max(int(c["embedding_dim"])//2, 64)),
    "Transformer": lambda c: TransformerLM(vocab_size=int(c["vocab_size"]), embedding_dim=int(c["embedding_dim"]), hidden_dim=int(c["hidden_dim"]), nhead=int(c["transformer_heads"]), num_layers=int(c["transformer_layers"])),
}

best_model_name = str(leaderboard.iloc[0]["model"])
cfg = registry[best_model_name]
vocab_path = Path(str(cfg.get("vocab_path", PROJECT_ROOT / "outputs" / "vocab.json")))
vocab = Vocabulary.load(vocab_path)

model = MODEL_BUILDERS[best_model_name](cfg)
checkpoint = torch.load(Path(cfg["checkpoint_path"]), map_location="cpu", weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])

engine = PredictiveKeyboardEngine(model=model, vocabulary=vocab, context_length=int(cfg["context_len"]), device="cpu")
print("Loaded model:", best_model_name)


In [ ]:

prompt = "I would like to"

print("Top-3 suggestions:")
for row in engine.predict(prompt, top_k=3):
    print(row)

print("\nTop-5 suggestions:")
for row in engine.predict(prompt, top_k=5):
    print(row)

print("\nBeam suggestions:")
for row in engine.predict(prompt, top_k=5, strategy="beam"):
    print(row)


In [ ]:

simulation = engine.simulate_keyboard_step("the meaning of", suggestion_count=3)
simulation



## Deployment
Run local app:

```bash
uv run streamlit run app/streamlit_app.py
```

App exposes top-3/top-5 suggestions, probabilities, autocomplete, and strategy controls.
